# Supervised Coding-Agent Harnesses

Build a reviewable harness around a proposed change and measure whether its checks catch regressions.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · Someone has to approve the formatter change

Tomas maintains a help-desk formatter that other teams depend on. He wants to accept small assistant-suggested changes while showing reviewers what changed, what was tested, and why a repeated request cannot apply twice.

The workplace scenario is fictional.


## Specify and verify a candidate


### Lecture 01 · Write an acceptance contract for a change

Turn a small change request into regression and acceptance examples.

#### Protect an old behavior on purpose

Whitespace should collapse in tickets. Empty inputs must still go to review. The first suggested change disregards the legacy behavior.

Write acceptance cases before implementation. Include a failing case that demonstrates the new requirement and a passing case for what people already expect.

#### How it works

An acceptance case states an observable input and expected output. Freeze current behaviors you want to preserve before asking for a change, and keep acceptance cases independent from any particular implementation. A small fixture makes a failing example cheap to inspect.

##### Write behavior before inviting implementation

An acceptance contract describes observable results for the proposed change, side effects, and old behavior the user still relies on. Build a small set that distinguishes the current implementation from the proposed one. A good starting set includes a normal input, a boundary case, and a preserved behavior.

For a formatter, “trim whitespace” leaves open whether internal spaces collapse and what happens when the result is empty. Use an input with both surrounding and internal spaces. An empty input can still go to the existing review mechanism. Test both so the proposed improvement cannot accidentally turn an unparseable ticket into an accepted one.

First run the tests against the current implementation. You expect the new case to fail for the reason you intend to fix, while existing cases pass. A test that already passes both implementations provides little evidence about the change. Keep the input and expected output in the review packet.

#### Worked example

A task-title formatter should now remove redundant spaces. Existing lowercasing and empty-title behavior still matter. Add a fixture for tabs and a case that captures the default title.

#### Try it

Write one legacy case and one case that fails with the baseline.


In [ ]:
def current_title(value):
    return value.strip().lower() or "untitled"


CASES = (
    (" Study PLAN ", "study plan"),
    ("", "untitled"),
    ("draft\t  release   plan", "draft release plan"),
)


def regressions(formatter):
    failures = []
    for value, expected in CASES:
        actual = formatter(value)
        if actual != expected:
            failures.append({"input": value, "expected": expected, "actual": actual})
    return failures


assert len(regressions(current_title)) == 1


**Compare your result**

The whitespace case should reproduce the new requirement. The legacy case should remain green after the change.


### Lecture 02 · Constrain proposals and run checks

Validate the shape and scope of a proposed change before running the acceptance suite.

#### A known proposal can still be wrong

A candidate uses a permitted operation but changes a part of the workflow outside the requested format. It passes a general style check.

Validate scope and run focused regression cases separately. Do not give an assistant arbitrary execution authority for a narrow formatting task. Give the reviewer the failing case and a small diff.

#### How it works

A model can propose a typed change request without receiving a shell or arbitrary file-writing capability. In the workbook, a proposal selects one registered formatter. The harness rejects unknown actions and runs the same fixtures against each candidate. The review record retains test details.

##### Check the proposal before granting effects

A harness can accept a small, typed proposal and run independent checks. Separate the declared operation and scope from the mechanism that applies it. An allowed field can still carry a change that violates the request, so validation of format and regression tests both matter.

A formatter proposal may name a recognized transformation and a known file. It can still change output for a legacy category outside the requested task. Evaluate the proposed behavior against the precise fixtures, and show the reviewer a diff small enough to inspect. A general lint pass adds evidence about style, which is weaker evidence about behavior.

Use a bounded sandbox or simulated effect for student exercises. Preserve the current state until a person has reviewed whatever needs approval. Record failures as evidence for the next attempt, such as a malformed proposal, an out-of-scope change, or a failing example. That makes review work measurable and prevents a retry from becoming blind expansion.

#### Worked example

A known but incorrect formatter lowercases text while preserving doubled spaces. It passes some existing behaviors and still fails the added requirement.

#### Try it

Evaluate an unknown proposal and a known incorrect candidate. Compare the failure categories.


In [ ]:
from dataclasses import dataclass


def proposed_title(value):
    return " ".join(value.lower().split()) or "untitled"


CANDIDATES = {"current": current_title, "collapse-spaces": proposed_title}


@dataclass(frozen=True)
class Review:
    candidate: str
    revision: int
    failures: tuple
    approved: bool = False


def evaluate(proposal, revision):
    if (
        not isinstance(proposal, dict) or set(proposal) != {"action", "candidate"}
        or proposal["action"] != "update-formatter" or not isinstance(proposal["candidate"], str)
        or proposal["candidate"] not in CANDIDATES
    ):
        raise ValueError("proposal outside the allowed contract")
    candidate = proposal["candidate"]
    return Review(candidate, revision, tuple(regressions(CANDIDATES[candidate])))


proposal = {"action": "update-formatter", "candidate": "collapse-spaces"}
assert evaluate(proposal, 1).failures == ()
assert evaluate(proposal | {"candidate": "current"}, 1).failures
try:
    evaluate(proposal | {"action": "unregistered-action"}, 1)
except ValueError:
    pass
else:
    raise AssertionError("unknown action accepted")


**Compare your result**

An unknown proposal fails validation. A known candidate should return concrete test failures for the reviewer.


## Approve and audit a change


### Lecture 03 · Approval, revisions and replay

Bind an approval to the reviewed proposal and current revision.

#### Approval expires when the reviewed payload changes

A reviewer approves a proposed revision in the morning. A later edit changes the payload before the worker applies it.

Bind approval to the exact reviewed revision and record a stable request identifier. A stale approval should cause a new review. A retry of an already applied version should return its recorded outcome.

#### How it works

Approval is a decision about a specific candidate and the state against which it was tested. Compare the reviewed revision with the current revision before applying the change. Store a stable request ID and outcome so retries can return the previous result without repeating the action.

##### Bind authorization to a specific revision

An approval for one revision is not general permission to apply later changes. Preserve the exact input, proposed effect, approval, and an operation identifier. Before applying, compare the current proposal with the approved revision. If it changed, ask for a new review.

Now suppose the worker applies a change but the acknowledgment is lost. A retry should return the recorded outcome when the system already completed that operation. Store completion alongside the relevant state change when the storage boundary permits it. An external action needs its own stable request identity and a way to reconcile uncertain outcomes.

Exercise two cases independently. Submitting a changed proposal after approval should wait. Resubmitting the completed operation should produce no second effect. Both cases can look like a duplicate request on arrival, but they need different handling. A trace should retain which revision and completion record the harness used.

#### Worked example

A reviewer approves the whitespace change at revision 1. Another edit lands before approval is applied. The old approval now needs a fresh review.

#### Try it

Try applying a good proposal before approval. Replay after applying it and then try the old review against a newer revision.


In [ ]:
from dataclasses import replace


class FormatterStore:
    def __init__(self):
        self.revision = 1
        self.candidate = "current"
        self.records = {}

    def apply(self, request_id, review):
        if request_id in self.records:
            recorded, revision = self.records[request_id]
            if recorded != review:
                raise ValueError("request ID reused with a different review")
            return "replayed", revision
        if review.revision != self.revision:
            return "stale", self.revision
        if review.failures:
            return "failed_checks", self.revision
        if not review.approved:
            return "awaiting_review", self.revision
        self.candidate = review.candidate
        self.revision += 1
        self.records[request_id] = review, self.revision
        return "applied", self.revision


store = FormatterStore()
review = evaluate(proposal, store.revision)
assert store.apply("change-1", review) == ("awaiting_review", 1)
approved = replace(review, approved=True)
assert store.apply("change-1", approved) == ("applied", 2)
assert store.apply("change-1", approved) == ("replayed", 2)
assert store.apply("change-2", approved) == ("stale", 2)
assert store.candidate == "collapse-spaces"


**Compare your result**

Return a waiting result before approval, the recorded result on replay, and a stale result when a different request uses an older review.


### Lecture 04 · Compare workflow traces and handoffs

Use a fixed suite to compare proposals and design a human handoff with useful evidence.

#### Make the result make sense in the morning

The on-call engineer finds that three proposals were blocked. A log with only ‘false’ leaves the team guessing why.

Record the proposed revision, check result, owner, and next action. Distinguish an invalid proposal from pending approval and an applied replay so another colleague can safely continue.

#### How it works

Count validated proposals, test passes, approval waits, and applications separately. A trace should be detailed enough to reproduce the decision and compact enough for a reviewer to scan. In a framework, carry the proposal ID and revision through each transition and keep the executor boundary explicit.

##### Tell the next operator what actually happened

A compact trace can show the request, proposal, checks, decision owner, applied revision, and outcome. It gives the reviewer a way to understand a blocked effect without replaying every intermediate generation. Organize traces by stable request identity so retries can be followed across process restarts.

Compare traces from two workflows at the decision points. One may spend less time generating changes but produce more rejected proposals. Another may wait longer for review but lower the rate of surprises after execution. Include total completion time, reviewer effort, and unresolved work when judging the workflow.

A handoff should distinguish “check failed and needs a revised proposal,” “passed and waiting for a human,” and “already applied.” Those states imply different next steps. Test that a new operator can pick up the right action using the evidence in the trace, without needing access to hidden model reasoning.

#### Worked example

Two candidates may look identical in a summary. A trace showing a failed whitespace case and a review approval against a specific revision distinguishes them.

#### Try it

Create a small report from the trace. Identify what the reviewer sees at each blocked stage.


In [ ]:
trace = []
scenario = FormatterStore()
for request_id, candidate_id, approve in (
    ("invalid-regression", "current", True),
    ("valid-change", "collapse-spaces", False),
    ("valid-change", "collapse-spaces", True),
):
    reviewed = replace(evaluate(proposal | {"candidate": candidate_id}, scenario.revision), approved=approve)
    outcome, revision = scenario.apply(request_id, reviewed)
    trace.append({
        "request": request_id, "candidate": candidate_id, "reviewed_revision": reviewed.revision,
        "test_failures": reviewed.failures, "decision": outcome, "resulting_revision": revision,
    })
assert [entry["decision"] for entry in trace] == ["failed_checks", "awaiting_review", "applied"]
for decision in trace:
    print(decision)


**Compare your result**

The report should name the failed check or approval state and the next responsible person. Keep the evidence tied to the exact candidate.


## Course project

Demonstrate a supervised change to a deterministic formatter. Preserve its spec, structured proposal, regression report, approval decision, and replay record.

Show that a wrong proposal fails before approval, an otherwise valid proposal waits, and a repeated approved request does not apply twice. Explain how your checks and review would transfer to a coding-agent framework.
